In [1]:
!pip install gradio

In [ ]:
 !unzip "/content/assign1_npx.zip"

In [ ]:
import gradio as gr
import numpy as np
import torch
import pickle
import PIL.Image
from pathlib import Path
import os
import sys
import time
import subprocess
import urllib.request
import glob
import functools

def setup_stylegan_repository():
    repo_path = "/content/stylegan2-ada-pytorch"

    if not Path(repo_path).exists():
        print("📥 StyleGAN2 repository not found. Cloning...")
        try:
            result = subprocess.run([
                "git", "clone",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git",
                repo_path
            ], check=True, capture_output=True, text=True)
            print("✅ Repository cloned successfully!")
        except subprocess.CalledProcessError as e:
            raise Exception(f"❌ Failed to clone repository: {e.stderr}")

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    return repo_path


def download_ffhq_model(repo_path):
    model_path = os.path.join(repo_path, 'ffhq.pkl')

    if not Path(model_path).exists():
        print("🔄 FFHQ model not found. Downloading...")
        try:
            url = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"
            urllib.request.urlretrieve(url, model_path)
            print("✅ Model downloaded successfully!")
        except Exception as e:
            raise Exception(f"❌ Failed to download model: {e}")

    return model_path


@functools.lru_cache(maxsize=1)
def load_stylegan_model():
    repo_path = setup_stylegan_repository()
    model_path = download_ffhq_model(repo_path)

    try:
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
            G = model['G_ema']

            if torch.cuda.is_available():
                G = G.cuda()
                print("🚀 Using GPU acceleration!")
            else:
                print("⚠️ GPU not available, using CPU")

        return G
    except Exception as e:
        raise Exception(f"❌ Failed to load StyleGAN2 model: {e}")


@functools.lru_cache(maxsize=1)
def load_latent_directions():
    np.random.seed(42)
    directions = {
        'smile': np.random.randn(1, 512) * 0.15,
        'age': np.random.randn(1, 512) * 0.12,
        'gender': np.random.randn(1, 512) * 0.18
    }
    return directions


@functools.lru_cache(maxsize=1)
def load_precomputed_latents():
    possible_paths = [
        "/content/assign1_npx",
        "/content/drive/MyDrive/assign1_npx",
        "/content/sample_data/assign1_npx",
        "assign1_npx"
    ]

    latent_folder = None
    for path in possible_paths:
        if os.path.exists(path):
            latent_folder = path
            break

    if latent_folder is None:
        raise Exception(f"❌ Folder 'assign1_npx' not found in any of these locations: {possible_paths}")

    print(f"📁 Found latents in: {latent_folder}")

    w_files = sorted(glob.glob(os.path.join(latent_folder, "w_latent_*.npy")))
    z_files = sorted(glob.glob(os.path.join(latent_folder, "z_latent_*.npy")))

    w_latents = []
    z_latents = []

    for w_file in w_files:
        try:
            w = np.load(w_file)
            w_latents.append(w)
        except Exception as e:
            print(f"Failed to load {w_file}: {e}")

    for z_file in z_files:
        try:
            z = np.load(z_file)
            z_latents.append(z)
        except Exception as e:
            print(f"Failed to load {z_file}: {e}")

    print(f"✅ Loaded {len(w_latents)} W latents and {len(z_latents)} Z latents")
    return w_latents, z_latents


def generate_image_from_latent(G, w_latent):
    if w_latent.ndim == 2:
        w_latent = w_latent[None, ...]
    elif w_latent.ndim != 3:
        return None

    device = next(G.parameters()).device
    w_tensor = torch.from_numpy(w_latent).to(device).float()

    with torch.no_grad():
        img = G.synthesis(w_tensor, noise_mode='const')

    img = img.to('cpu')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).numpy()[0]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return img


def apply_directions(base_w, directions, smile_val, age_val, gender_val):
    modified_w = base_w.copy()
    modified_w = modified_w + directions['smile'] * smile_val
    modified_w = modified_w + directions['age'] * age_val
    modified_w = modified_w + directions['gender'] * gender_val
    return modified_w


def generate_face(face_idx, smile, age, gender, progress=gr.Progress()):
    progress(0, desc="Loading models...")

    try:
        G = load_stylegan_model()
        directions = load_latent_directions()
        w_latents, z_latents = load_precomputed_latents()

        if not w_latents or len(w_latents) == 0:
            return None, "❌ No precomputed latents found!", ""

        progress(0.3, desc="Applying transformations...")

        base_w = w_latents[face_idx]

        current_w = apply_directions(base_w, directions, smile, age, gender)

        progress(0.7, desc="Generating image...")

        start_time = time.time()
        img = generate_image_from_latent(G, current_w)
        generation_time = time.time() - start_time

        progress(1.0, desc="Complete!")

        if img is not None:
            perf_color = "🟢" if generation_time < 0.5 else "🟡" if generation_time < 1.0 else "🔴"
            status = f"{perf_color} Generated in {generation_time:.3f}s"
            info = f"Face {face_idx + 1}/{len(w_latents)} | Smile: {smile:.1f} | Age: {age:.1f} | Gender: {gender:.1f}"

            return img, status, info
        else:
            return None, "❌ Failed to generate image", ""

    except Exception as e:
        return None, f"❌ Error: {str(e)}", ""


def save_face(face_idx, smile, age, gender):
    try:
        G = load_stylegan_model()
        directions = load_latent_directions()
        w_latents, z_latents = load_precomputed_latents()

        if not w_latents or len(w_latents) == 0:
            return "❌ No precomputed latents found!"

        timestamp = int(time.time())
        save_dir = "/content/saved_faces"
        os.makedirs(save_dir, exist_ok=True)

        base_w = w_latents[face_idx]
        current_w = apply_directions(base_w, directions, smile, age, gender)

        np.save(f"{save_dir}/edited_w_latent_{timestamp}.npy", current_w)

        img = generate_image_from_latent(G, current_w)
        if img is not None:
            img_pil = PIL.Image.fromarray(img)
            img_pil.save(f"{save_dir}/edited_face_{timestamp}.png")
            return f"💾 Saved to /content/saved_faces/edited_face_{timestamp}.png"
        else:
            return "❌ Failed to generate image for saving"

    except Exception as e:
        return f"❌ Save failed: {str(e)}"


def get_system_info():
    device_type = "GPU" if torch.cuda.is_available() else "CPU"
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    repo_status = "✅ Ready" if Path('/content/stylegan2-ada-pytorch').exists() else "❌ Missing"

    try:
        w_latents, z_latents = load_precomputed_latents()
        w_count = len(w_latents) if w_latents else 0
        z_count = len(z_latents) if z_latents else 0
    except:
        w_count = 0
        z_count = 0

    info = f"""
**System Status:**
- **Device**: {device_type} ({gpu_name if torch.cuda.is_available() else 'CPU'})
- **Repository**: {repo_status}
- **W Latents**: {w_count} files
- **Z Latents**: {z_count} files
"""
    return info


def create_interface():

    with gr.Blocks(
        title="🎭 Real-Time Latent Editing Interface",
        theme=gr.themes.Soft(),
        css="""
        .gradio-container {
            max-width: 1200px !important;
        }
        .status-text {
            font-family: monospace;
            font-size: 14px;
        }
        """
    ) as demo:

        gr.Markdown("# 🎭 Real-Time Latent Editing Interface")
        gr.Markdown("Interactive facial feature manipulation using pre-computed StyleGAN2 latents")

        system_info = gr.Markdown(get_system_info())

        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                gr.Markdown("## 🎮 Controls")

                face_idx = gr.Slider(
                    minimum=0,
                    maximum=49,
                    step=1,
                    value=0,
                    label="🎭 Select Face",
                    info="Choose which face to edit"
                )

                with gr.Row():
                    prev_btn = gr.Button("⬅️ Previous", size="sm")
                    next_btn = gr.Button("➡️ Next", size="sm")
                    random_btn = gr.Button("🎲 Random", size="sm")

                with gr.Row():
                    reset_btn = gr.Button("🔄 Reset", variant="secondary")
                    save_btn = gr.Button("💾 Save", variant="primary")

                gr.Markdown("---")

                gr.Markdown("## 🎨 Attributes")

                smile = gr.Slider(
                    minimum=-3.0,
                    maximum=3.0,
                    value=0.0,
                    step=0.2,
                    label="😊 Smile",
                    info="Adjust smile intensity"
                )

                age = gr.Slider(
                    minimum=-3.0,
                    maximum=3.0,
                    value=0.0,
                    step=0.2,
                    label="👴 Age",
                    info="Adjust apparent age"
                )

                gender = gr.Slider(
                    minimum=-3.0,
                    maximum=3.0,
                    value=0.0,
                    step=0.2,
                    label="⚧️ Gender",
                    info="Adjust gender characteristics"
                )

            with gr.Column(scale=2):
                gr.Markdown("## 🖼️ Generated Face")

                image_output = gr.Image(
                    label="Generated Face",
                    width=512,
                    height=512,
                    interactive=False
                )

                status_text = gr.Markdown("Ready to generate...", elem_classes=["status-text"])
                info_text = gr.Markdown("", elem_classes=["status-text"])
                save_status = gr.Markdown("")

        with gr.Accordion("ℹ️ Instructions & Tips", open=False):
            gr.Markdown("")

        def update_face_range():
            try:
                w_latents, _ = load_precomputed_latents()
                if w_latents:
                    return gr.Slider(maximum=len(w_latents)-1)
            except:
                pass
            return gr.Slider(maximum=49)

        def navigate_face(current_idx, direction):
            try:
                w_latents, _ = load_precomputed_latents()
                if not w_latents:
                    return current_idx, 0.0, 0.0, 0.0

                max_idx = len(w_latents) - 1

                if direction == "prev":
                    new_idx = max(0, current_idx - 1)
                elif direction == "next":
                    new_idx = min(max_idx, current_idx + 1)
                else:
                    new_idx = np.random.randint(0, len(w_latents))

                return new_idx, 0.0, 0.0, 0.0
            except:
                return current_idx, 0.0, 0.0, 0.0

        inputs = [face_idx, smile, age, gender]
        outputs = [image_output, status_text, info_text]

        for component in inputs:
            component.change(
                fn=generate_face,
                inputs=inputs,
                outputs=outputs
            )

        prev_btn.click(
            fn=lambda idx, s, a, g: navigate_face(idx, "prev"),
            inputs=inputs,
            outputs=inputs
        )

        next_btn.click(
            fn=lambda idx, s, a, g: navigate_face(idx, "next"),
            inputs=inputs,
            outputs=inputs
        )

        random_btn.click(
            fn=lambda idx, s, a, g: navigate_face(idx, "random"),
            inputs=inputs,
            outputs=inputs
        )

        reset_btn.click(
            fn=lambda idx, s, a, g: (idx, 0.0, 0.0, 0.0),
            inputs=inputs,
            outputs=inputs
        )

        save_btn.click(
            fn=save_face,
            inputs=inputs,
            outputs=[save_status]
        )

        demo.load(
            fn=get_system_info,
            outputs=[system_info]
        )

    return demo


if __name__ == "__main__":
    demo = create_interface()
    demo.launch(
        share=True,
        debug=True,
        server_port=7860
    )

📁 Found latents in: /content/assign1_npx
✅ Loaded 10 W latents and 10 Z latents
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://98c0d559a21905b9cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 StyleGAN2 repository not found. Cloning...
✅ Repository cloned successfully!
🔄 FFHQ model not found. Downloading...
✅ Model downloaded successfully!
🚀 Using GPU acceleration!
Setting up PyTorch plugin "bias_act_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py:50: UserWarning: Failed to build CUDA kernels for bias_act. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py", line 48, in _init
    _plugin = custom_ops.get_plugin('bias_act_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1798, in _jit_compile
    _write_ninja_file_and_

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

...

/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo